In [0]:
from pyspark.sql import functions as F

start_date = "2017-07-01" #first order
end_date   = "2021-12-31" #last target
#future improvements:link the strat date with sales(2017), link the end_date with targets
date_gold = spark.sql(f"""SELECT explode(sequence(to_date('{start_date}'), to_date('{end_date}'), interval 1 day)) AS date""")

dim_date = date_gold.select(
    F.date_format("date", "yyyyMMdd").cast("int").alias("date_key"),
    F.col("date").alias("date"),
    F.year("date").alias("year"),
    F.month("date").alias("month"),
    F.dayofmonth("date").alias("day"),
    F.date_format("date", "yyyyMM").cast("int").alias("year_month"),
    F.date_format("date", "MMMM").alias("month_name"),
    F.date_format("date", "EEEE").alias("day_name"),
    F.when(F.dayofweek("date").isin(1, 7), 1).otherwise(0).alias("IsWeekend"),
    F.trunc("date", "MM").alias("month_start_date") 
    #transforming the date to the first day of the month(15/05/2023 -> 01/05/2023) so we can join with fact_target
)

(dim_date.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true") #if we want to change any column later
    .saveAsTable("sales_lakehouse.gold.dim_date"))


In [0]:
dim_date.printSchema()

In [0]:
display(dim_date)

In [0]:
duplicate_count = dim_date.groupBy("date_key").count().filter("count > 1").count()

if duplicate_count == 0:
    print("No dups found")
else:
    print(f"{duplicate_count} duplicates found")

In [0]:
null_check = dim_date\
.select([F.count(F.when(F.col(c).isNull(), c))
         .alias(c) for c in dim_date.columns])

null_check.show()

In [0]:
targets_without_dates = spark.sql("""
    SELECT DISTINCT t.date_key 
    FROM sales_lakehouse.gold.fact_targets t
    LEFT JOIN sales_lakehouse.gold.dim_date d ON t.date_key = d.date_key
    WHERE d.date_key IS NULL
""")

if targets_without_dates.count() == 0:
    print("All targets are linked with a date")
else:
    print("Some targets have date_keys that don't exist in Dim_Date!")
    targets_without_dates.show()

In [0]:
duplicate_targets_per_employee = spark.sql("""
    SELECT employee_key, date_key, COUNT(*) as count
    FROM sales_lakehouse.gold.fact_targets
    GROUP BY employee_key, date_key
    HAVING COUNT(*) > 1
""")

if duplicate_targets_per_employee.count() == 0:
    print("No duplicate targets per employee/month.")
else:
    print("Found duplicate targets for the same employee in the same month!")
    duplicate_targets_per_employee.show()

In [0]:
missing_employees = spark.sql("""
    SELECT DISTINCT t.employee_key 
    FROM sales_lakehouse.gold.fact_targets t
    LEFT JOIN sales_lakehouse.gold.dim_salesperson s ON t.employee_key = s.employee_key
    WHERE s.employee_key IS NULL
""")

if missing_employees.count() == 0:
    print("All targets belong to a valid Salesperson.")
else:
    print("Found targets for Employees that do not exist in the dim_alesperson")
    missing_employees.show()